# PSM Memory Fine-Tune Notebook

This notebook prepares the first PSM memory fine-tuning dataset and runs a LoRA SFT job. It is intentionally conservative: generate local examples first, validate schema, upload artifacts, then train a Qwen 1.5B-compatible adapter.

Use FP16/full model behavior as the reference, but evaluate deployable quantizations separately after training.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os

RUN_LABEL = "psm-memory-sft-v1"
RUN_ID = f"{RUN_LABEL}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

# Repos. Change these before upload/training if needed.
SOURCE_REPO_URL = "https://github.com/chkrishna2001/PSM.git"
HF_DATASET_REPO = "chkrishna2001/nano-psm"
HF_ADAPTER_REPO = f"chkrishna2001/{RUN_ID}"

# Base model for LoRA training. Keep this aligned with the model family used to produce the GGUF.
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Data generation.
GENERATE_LIMIT = 500
VALIDATION_RATIO = 0.15
WORK_DIR = Path("/content/psm-memory-fine-tune")
REPO_DIR = Path("/content/PSM")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "adapter"

# Training knobs. Start small; increase after schema quality is proven.
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

print(f"RUN_ID={RUN_ID}")
print(f"HF_DATASET_REPO={HF_DATASET_REPO}")
print(f"HF_ADAPTER_REPO={HF_ADAPTER_REPO}")

In [ ]:
# Install system and Python dependencies.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - >/tmp/node_setup.log
!sudo apt-get install -y nodejs >/tmp/apt_node.log
!pip -q install -U huggingface_hub datasets transformers accelerate peft bitsandbytes sentencepiece
!node --version
!python --version

In [ ]:
# Authenticate to Hugging Face for private dataset/model repos.
import getpass
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN') or (userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_TOKEN') or '')
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(HF_DATASET_REPO, repo_type='dataset', private=True, exist_ok=True)
api.create_repo(HF_ADAPTER_REPO, repo_type='model', private=True, exist_ok=True)

In [ ]:
# Clone current PSM and generate supervised examples.
!rm -rf "$REPO_DIR" "$WORK_DIR"
!git clone --depth 1 "$SOURCE_REPO_URL" "$REPO_DIR"
!mkdir -p "$DATA_DIR"
%cd /content/PSM
!node nano-psm/data-pipeline/src/generate-dataset.mjs --locomo benchmark/locomo/data/locomo10.json --out "$DATA_DIR" --limit $GENERATE_LIMIT --validation-ratio $VALIDATION_RATIO
!node nano-psm/data-pipeline/src/validate-examples.mjs "$DATA_DIR/train.jsonl" "$DATA_DIR/validation.jsonl"
!ls -lh "$DATA_DIR"

In [ ]:
# Upload exact training artifacts for reproducibility.
from huggingface_hub import upload_folder
DATASET_RUN_DIR = f"runs/{RUN_ID}"
upload_folder(
    repo_id=HF_DATASET_REPO,
    repo_type='dataset',
    folder_path=str(DATA_DIR),
    path_in_repo=DATASET_RUN_DIR,
    commit_message=f"add {RUN_ID} fine-tune dataset",
    token=os.environ['HF_TOKEN'],
)
print(f"Uploaded dataset to hf://{HF_DATASET_REPO}/{DATASET_RUN_DIR}")

In [ ]:
# Load JSONL and format examples as instruction/input/output text.
import json
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': str(DATA_DIR / 'train.jsonl'),
    'validation': str(DATA_DIR / 'validation.jsonl'),
})

SYSTEM = "You are PSM, a memory-management model. Return JSON only using the required schema."

def format_example(row):
    input_json = json.dumps(row['input'], ensure_ascii=False, separators=(',', ':'))
    output_json = json.dumps(row['output'], ensure_ascii=False, separators=(',', ':'))
    return {
        'text': (
            f"<|system|>\n{SYSTEM}\n"
            f"<|user|>\n{row['instruction']}\nInput JSON:\n{input_json}\n"
            f"<|assistant|>\n{output_json}"
        )
    }

dataset = dataset.map(format_example, remove_columns=dataset['train'].column_names)
print(dataset)
print(dataset['train'][0]['text'][:1200])

In [ ]:
# Load base model in 4-bit for LoRA training.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Tokenize full text. This trains the model to emit the expected assistant JSON after the prompt.
def tokenize(batch):
    tokenized = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    tokenized['labels'] = [ids.copy() for ids in tokenized['input_ids']]
    return tokenized

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(tokenized)

In [ ]:
# Train adapter.
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    data_collator=collator,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

In [ ]:
# Publish the LoRA adapter. Merge/quantize in a separate notebook or job after validation.
upload_folder(
    repo_id=HF_ADAPTER_REPO,
    repo_type='model',
    folder_path=str(OUTPUT_DIR),
    path_in_repo='.',
    commit_message=f"add {RUN_ID} LoRA adapter",
    token=os.environ['HF_TOKEN'],
)
print(f"Uploaded adapter to hf://{HF_ADAPTER_REPO}")

## After Training

Do not judge this adapter only by training loss. Run the validation prompts through FP16, Q8, Q6, Q5, and Q4 exports and score:

- valid JSON rate
- expected schema rate
- generic `User` leakage
- current-turn grounding
- unsupported fact rate
- temporal extraction accuracy
- ignore accuracy